#ATENÇÃO: PARA ESSA ETAPA QUEREMOS TANTO OS DADOS PARA PUBLICAÇÂO (842) QUANTO OS DADOS GEOREFERENCIADOS PARA A PRÓXIMA ETAPA E OS DADOS DE TODOS OS MUNICÍPIOS (2365)!

# 11j — Exportação de bases consolidadas para publicação

**Objetivo:** preparar duas bases analíticas para disponibilização pública (Zenodo, OSF, ou repositório institucional), em formato citável e reproduzível.

**Duas bases:**
1. **`renovabio_psm_cross_section.csv/parquet`** — Cross-section dos 842 canavieiros + universo CS comparativo (~2.000 munis), com baseline 2015-2019 das covariáveis PSM, flag de tratamento, propensity scores estimados.
2. **`renovabio_outcomes_panel.csv/parquet`** — Painel longitudinal 842 munis × 2012-2024, com todos os outcomes SEEG (totais e sub-canais), PAM (áreas e produção), MapBiomas (shares de uso), e identificadores.

**Acompanha:**
- `CODEBOOK_PSM.md` — descrição de todas as colunas da base PSM
- `CODEBOOK_PANEL.md` — descrição de todas as colunas da base painel
- `README.md` — visão geral, propósito, citação, licença, contato

**Pré-condições:**
- `panel_canavieiro_main.csv`, `seeg_subcanais_panel.csv`, `pam_1612_long_2012_2024.parquet`, `07_mapbiomas_panel_balanced_2015_2024_CORRIGIDO.csv`
- `base_psm_integrada_raw.csv`
- `psm_scores_FULL2.csv` (se houver)
- `share_cana_eq52_pre2018.csv`

## Setup

In [1]:
# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
EXPORT_DIR = BASE_DIR / "data" / "published"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Bases serão exportadas para: {EXPORT_DIR}")

Mounted at /content/drive
Bases serão exportadas para: /content/drive/MyDrive/Renovabio - EcoEco/data/published


In [2]:
import numpy as np
import pandas as pd
from pipeline.config import interim, out_pre

# Versão dos dados (incrementar a cada publicação)
DATA_VERSION = "v1.0"
RELEASE_DATE = "2026-05-22"

print(f"Versão dos dados: {DATA_VERSION} ({RELEASE_DATE})")

Versão dos dados: v1.0 (2026-05-22)


## Bloco 1 — Base PSM cross-section

Estrutura: 1 linha por município, identificadores + baseline covariáveis + tratamento + scores.

**Universo:** 2.018 municípios do Centro-Sul (SP, GO, MG, MS, PR, MT) que passaram pelos filtros B1. Inclui canavieiros (n=842) e não-canavieiros para comparação.

In [3]:
# Carregar fonte primária PSM
psm_raw = pd.read_csv(
    BASE_DIR / "data/raw/psm_baseline/base_psm_integrada_raw.csv",
    low_memory=False,
)
psm_raw["geocode"] = psm_raw["0_cd_ibge"].astype(str).str.zfill(7)
print(f"PSM raw: {psm_raw.shape}")

# Carregar painel canavieiro para identificar tratados e canavieiros
panel = pd.read_csv(interim("panel_canavieiro_main.csv"), dtype={"geocode": str})

# Identificadores municipais
muni_id = (panel.groupby("geocode", as_index=False)
           .agg(municipality=("municipio","first"), uf=("uf","first"),
                bioma=("bioma","first"),
                is_canavieiro=("geocode", lambda x: True),
                is_treated_ever=("is_treated_ever","first"),
                treatment_year=("g_m","first")))

print(f"Canavieiros identificados: {muni_id.shape[0]}")

PSM raw: (5570, 166)
Canavieiros identificados: 842


In [4]:
# Construir base PSM cross-section
def safe_log1p(s, idx):
    s = pd.to_numeric(s, errors="coerce") if s is not None else pd.Series(np.nan, index=idx)
    return np.log1p(s.clip(lower=0))
def safe_div(num, den, idx):
    num = pd.to_numeric(num, errors="coerce") if num is not None else pd.Series(np.nan, index=idx)
    den = pd.to_numeric(den, errors="coerce") if den is not None else pd.Series(np.nan, index=idx)
    return np.where((den.notna()) & (den > 0), num / den, np.nan)
def asn(s, idx):
    return pd.to_numeric(s, errors="coerce") if s is not None else pd.Series(np.nan, index=idx)

d = psm_raw.copy()
idx = d.index

psm_export = pd.DataFrame({
    "ibge_code": d["geocode"],
    "uf": d.get("0_uf", pd.Series([np.nan]*len(d))),
    # Demografia e economia (todas baseline 2015-2017 ou 2017)
    "log_pib_total": safe_log1p(d.get("1_pib_total"), idx),
    "log_pib_per_capita": safe_log1p(d.get("1_pib_percap"), idx),
    "log_population_2017": safe_log1p(d.get("2_pop_2017_ibge"), idx),
    "log_total_area_ha": safe_log1p(d.get("14_area_total"), idx),
    "population_density": safe_div(d.get("2_pop_2017_ibge"), d.get("14_area_total"), idx),
    # VABC shares (composição econômica)
    "share_vabc_agriculture": safe_div(d.get("1_vadc_agro"), d.get("1_vadc_bruto"), idx),
    "share_vabc_industry": safe_div(d.get("1_vadc_ind"), d.get("1_vadc_bruto"), idx),
    "share_vabc_services": safe_div(d.get("1_vadc_serv"), d.get("1_vadc_bruto"), idx),
    "share_vabc_public_admin": safe_div(d.get("1_vadc_adm"), d.get("1_vadc_bruto"), idx),
    # MapBiomas baseline 2015-2017 (proporções de cobertura)
    "mb_share_sugarcane_baseline": asn(d.get("3_mb_sharegrp_pre_cana"), idx),
    "mb_share_soybean_baseline": asn(d.get("3_mb_sharegrp_pre_soja"), idx),
    "mb_share_pasture_baseline": asn(d.get("3_mb_sharegrp_pre_pastagem"), idx),
    "mb_share_native_veg_baseline": asn(d.get("3_mb_sharegrp_pre_vegetacao_nativa"), idx),
    "mb_share_urban_baseline": asn(d.get("3_mb_sharegrp_pre_urbano_infra"), idx),
    "mb_share_silviculture_baseline": asn(d.get("3_mb_sharegrp_pre_silvicultura"), idx),
    "mb_share_agriculture_total_baseline": asn(d.get("3_mb_sharegrp_pre_agricultura_total"), idx),
    # PAM baseline (áreas colhidas)
    "log_sugarcane_area_ha": safe_log1p(d.get("4_area_colhida_ha_cana"), idx),
    "log_soybean_area_ha": safe_log1p(d.get("4_area_colhida_ha_soja"), idx),
    "log_maize_area_ha": safe_log1p(d.get("4_area_colhida_ha_milho"), idx),
    "log_cotton_area_ha": safe_log1p(d.get("4_area_colhida_ha_alg"), idx),
    "log_agriculture_total_area_ha": safe_log1p(d.get("4_area_colhida_ha"), idx),
    "share_sugarcane_of_agriculture": safe_div(d.get("4_area_colhida_ha_cana"), d.get("4_area_colhida_ha"), idx),
    # Censo agropecuário 2017 — estrutura fundiária
    "share_family_farms": safe_div(d.get("5_num_est_af"), d.get("5_num_est_total"), idx),
    "share_medium_large_farms": safe_div(d.get("5_num_est_mp"), d.get("5_num_est_total"), idx),
    "share_family_farms_area": safe_div(d.get("6_area_lav_af"), d.get("6_area_lav_total"), idx),
    "share_medium_large_farms_area": safe_div(d.get("6_area_lav_mp"), d.get("6_area_lav_total"), idx),
    # Mecanização e infraestrutura
    "tractors_per_farm": safe_div(d.get("11_num_trator_total"), d.get("5_num_est_total"), idx),
    "share_irrigated_farms": safe_div(d.get("12_num_est_irrig_total"), d.get("5_num_est_total"), idx),
    "share_irrigated_area": safe_div(d.get("12_area_irrig_total"), d.get("6_area_lav_total"), idx),
    "share_financed_farms": safe_div(d.get("13_num_est_fin_total"), d.get("5_num_est_total"), idx),
    "share_tech_assistance": safe_div(d.get("10_num_est_receb_at"), d.get("5_num_est_total"), idx),
    "share_natural_vegetation_area": safe_div(d.get("14_vegetacao_natural"), d.get("14_area_total"), idx),
    "pct_farms_with_energy": asn(d.get("7_est_com_energia%"), idx),
    # Indicadores sociais
    "hdim_education": asn(d.get("17_idhm_educ"), idx),
    "hdim_income": asn(d.get("17_idhm_renda"), idx),
    "hdim_longevity": asn(d.get("17_idhm_long"), idx),
    "social_vulnerability_infrastructure": asn(d.get("17_ivs_infraestrutura_urbana"), idx),
    "social_vulnerability_human_capital": asn(d.get("17_ivs_capital_humano"), idx),
    "social_vulnerability_income_work": asn(d.get("17_ivs_renda_e_trabalho"), idx),
    "gini_index": asn(d.get("17_i_gini"), idx),
})

# Clipagem de shares para [0, 1] (mesma lógica do pipeline interno)
share_cols = [c for c in psm_export.columns if c.startswith("share_") or c.startswith("mb_share_")]
for c in share_cols:
    s = pd.to_numeric(psm_export[c], errors="coerce")
    if not s.dropna().empty and (s.dropna().between(-0.05, 1.05).mean() > 0.8):
        psm_export[c] = s.clip(0, 1)

print(f"Base PSM construída: {psm_export.shape}")
print(f"  munis: {psm_export['ibge_code'].nunique()}")

Base PSM construída: (5570, 42)
  munis: 5570


In [5]:
# Adicionar status canavieiro + tratamento
psm_export = psm_export.merge(
    muni_id.rename(columns={"geocode": "ibge_code"})[
        ["ibge_code", "municipality", "bioma", "is_canavieiro",
         "is_treated_ever", "treatment_year"]
    ],
    on="ibge_code", how="left",
)
psm_export["is_canavieiro"] = psm_export["is_canavieiro"].fillna(False)
psm_export["is_treated_ever"] = psm_export["is_treated_ever"].fillna(False)

# Reordenar colunas para identificadores no início
cols_id = ["ibge_code", "municipality", "uf", "bioma",
           "is_canavieiro", "is_treated_ever", "treatment_year"]
cols_rest = [c for c in psm_export.columns if c not in cols_id]
psm_export = psm_export[cols_id + cols_rest]

# Adicionar propensity scores se disponíveis
psm_scores_path = interim("psm_scores_FULL2.csv")
if not psm_scores_path.exists():
    candidates = list((BASE_DIR / "data" / "interim").glob("psm_scores*.csv"))
    if candidates:
        psm_scores_path = candidates[0]

if psm_scores_path.exists():
    ps = pd.read_csv(psm_scores_path, dtype={"geocode": str})
    ps = ps.rename(columns={"geocode": "ibge_code", "pscore": "propensity_score_full2"})
    keep = ["ibge_code"] + [c for c in ps.columns if "pscore" in c.lower()
                            or "score" in c.lower() or "matched" in c.lower()]
    keep = list(dict.fromkeys(keep))  # remove duplicatas mantendo ordem
    keep = [c for c in keep if c in ps.columns]
    psm_export = psm_export.merge(ps[keep], on="ibge_code", how="left")
    print(f"OK propensity scores adicionados de {psm_scores_path.name}")
else:
    print("AVISO: psm_scores não encontrado, propensity_score_full2 fica ausente")

print(f"\nBase PSM final: {psm_export.shape}")
print(f"  Tratados: {psm_export['is_treated_ever'].sum()}")
print(f"  Canavieiros (total): {psm_export['is_canavieiro'].sum()}")
print(f"  Universo CS: {len(psm_export)}")

AVISO: psm_scores não encontrado, propensity_score_full2 fica ausente

Base PSM final: (5570, 47)
  Tratados: 194
  Canavieiros (total): 842
  Universo CS: 5570


/tmp/ipykernel_2425/1068747620.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  psm_export["is_canavieiro"] = psm_export["is_canavieiro"].fillna(False)
/tmp/ipykernel_2425/1068747620.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  psm_export["is_treated_ever"] = psm_export["is_treated_ever"].fillna(False)


In [6]:
# Salvar
out_csv = EXPORT_DIR / f"renovabio_psm_cross_section_{DATA_VERSION}.csv"
out_pq = EXPORT_DIR / f"renovabio_psm_cross_section_{DATA_VERSION}.parquet"
psm_export.to_csv(out_csv, index=False, encoding="utf-8")
psm_export.to_parquet(out_pq, index=False)
print(f"OK salvos:")
print(f"  {out_csv.name}  ({out_csv.stat().st_size / 1024:.0f} KB)")
print(f"  {out_pq.name}  ({out_pq.stat().st_size / 1024:.0f} KB)")

OK salvos:
  renovabio_psm_cross_section_v1.0.csv  (2941 KB)
  renovabio_psm_cross_section_v1.0.parquet  (1245 KB)


## Bloco 2 — Base outcomes painel

Estrutura: 1 linha por município × ano, formato long, 842 canavieiros × 13 anos (2012-2024) = 10.946 linhas potenciais.

**Inclui:**
- SEEG totais (queima, solos manejados, LUC, carbono solo)
- SEEG sub-canais decompostos (cana_direto = res_cana + org_cana, fert_n, calagem, res_outros)
- PAM áreas e produção temporal (4 culturas: cana, soja, milho, algodão)
- MapBiomas shares anuais (6 classes)
- Identificadores e flag de tratamento por ano

In [7]:
# Base painel canônico (842 munis × 13 anos)
panel = pd.read_csv(interim("panel_canavieiro_main.csv"), dtype={"geocode": str})
print(f"Painel canônico: {panel.shape}")
print(f"  munis: {panel['geocode'].nunique()}, anos: {panel['ano'].nunique()}")

# Sub-canais SEEG
seeg_sub = pd.read_csv(interim("seeg_subcanais_panel.csv"))
seeg_sub["geocode"] = seeg_sub["geocode"].astype(str).str.zfill(7)
print(f"SEEG sub-canais: {seeg_sub.shape}")

Painel canônico: (8420, 139)
  munis: 842, anos: 10
SEEG sub-canais: (23630, 19)


In [8]:
# PAM long temporal
import os
pam_path = interim("pam_1612_long_2012_2024.parquet")
if pam_path.exists():
    pam = pd.read_parquet(pam_path)
    pam["geocode"] = pam["geocode"].astype(str).str.zfill(7) if "geocode" in pam.columns else pam["cod_ibge"].astype(str).str.zfill(7)
    print(f"PAM long: {pam.shape}")
    print(f"  culturas: {pam['cultura'].unique() if 'cultura' in pam.columns else 'check'}")
else:
    print("AVISO: PAM long não encontrado, painel sem PAM temporal")
    pam = None

# MapBiomas painel
mapb_path = interim("07_mapbiomas_panel_balanced_2015_2024_CORRIGIDO.csv")
if mapb_path.exists():
    mapb = pd.read_csv(mapb_path, dtype={"geocode": str})
    print(f"MapBiomas painel: {mapb.shape}")
else:
    print("AVISO: MapBiomas painel não encontrado")
    mapb = None

PAM long: (289224, 11)
  culturas: check
MapBiomas painel: (23630, 33)


In [9]:
# Construir painel exportável
panel_export = pd.DataFrame({
    "ibge_code": panel["geocode"],
    "year": panel["ano"].astype(int),
    "municipality": panel["municipio"],
    "uf": panel["uf"],
    "bioma": panel.get("bioma"),
    "is_treated_ever": panel["is_treated_ever"],
    "treatment_year": panel["g_m"],
    "is_post_treatment": (panel["ano"] >= panel["g_m"]).fillna(False),
    # SEEG totais (tCO2e/ano)
    "emissions_burning_tco2e": panel.get("queima"),
    "emissions_soils_managed_tco2e": panel.get("solos_manejados"),
    "emissions_luc_tco2e": panel.get("luc"),
    "emissions_carbon_soil_tco2e": panel.get("carbono_solo"),
})

# Join sub-canais SEEG
seeg_keep = ["geocode", "ano", "res_cana", "org_cana", "fert_n", "calagem", "res_outros"]
seeg_keep = [c for c in seeg_keep if c in seeg_sub.columns]
seeg_sub_r = seeg_sub[seeg_keep].rename(columns={
    "geocode": "ibge_code", "ano": "year",
    "res_cana": "emissions_sugarcane_residues_tco2e",
    "org_cana": "emissions_sugarcane_organic_tco2e",
    "fert_n": "emissions_n_fertilizer_tco2e",
    "calagem": "emissions_liming_tco2e",
    "res_outros": "emissions_other_residues_tco2e",
})
panel_export = panel_export.merge(seeg_sub_r, on=["ibge_code", "year"], how="left")

# Derivar cana_direto = res_cana + org_cana (consolidação v2.4)
panel_export["emissions_sugarcane_direct_tco2e"] = (
    panel_export["emissions_sugarcane_residues_tco2e"].fillna(0)
    + panel_export["emissions_sugarcane_organic_tco2e"].fillna(0)
)

print(f"Após SEEG: {panel_export.shape}")

Após SEEG: (8420, 18)


In [10]:
# Join PAM se disponível
if pam is not None:
    # Pivot PAM long para wide por cultura (área e produção)
    pam_wide_area = pam.pivot_table(
        index=["geocode", "ano"], columns="cultura",
        values="area_colhida_ha", aggfunc="first"
    ).reset_index()
    pam_wide_area = pam_wide_area.rename(columns={"geocode": "ibge_code", "ano": "year"})

    # Renomear culturas para inglês
    cultura_map = {
        "cana_de_acucar": "sugarcane", "cana-de-açúcar": "sugarcane",
        "soja": "soybean", "milho": "maize", "algodao": "cotton",
        "algodão": "cotton",
    }
    new_cols = {}
    for c in pam_wide_area.columns:
        if c not in ["ibge_code", "year"]:
            eng = cultura_map.get(c.lower(), c.lower())
            new_cols[c] = f"area_{eng}_ha"
    pam_wide_area = pam_wide_area.rename(columns=new_cols)
    panel_export = panel_export.merge(pam_wide_area, on=["ibge_code", "year"], how="left")

    # Produção
    if "quantidade_produzida_t" in pam.columns or "qtd_produzida" in pam.columns:
        prod_col = "quantidade_produzida_t" if "quantidade_produzida_t" in pam.columns else "qtd_produzida"
        pam_wide_prod = pam.pivot_table(
            index=["geocode", "ano"], columns="cultura",
            values=prod_col, aggfunc="first"
        ).reset_index()
        pam_wide_prod = pam_wide_prod.rename(columns={"geocode": "ibge_code", "ano": "year"})
        new_cols = {}
        for c in pam_wide_prod.columns:
            if c not in ["ibge_code", "year"]:
                eng = cultura_map.get(c.lower(), c.lower())
                new_cols[c] = f"production_{eng}_t"
        pam_wide_prod = pam_wide_prod.rename(columns=new_cols)
        panel_export = panel_export.merge(pam_wide_prod, on=["ibge_code", "year"], how="left")

# Join MapBiomas
if mapb is not None:
    mb_keep = ["geocode", "ano"]
    mb_rename = {"geocode": "ibge_code", "ano": "year"}
    classes_mb = {
        "share_cana": "mb_share_sugarcane",
        "share_pastagem": "mb_share_pasture",
        "share_vegetacao_nativa": "mb_share_native_veg",
        "share_soja": "mb_share_soybean",
        "share_silvicultura": "mb_share_silviculture",
        "share_urbano_infra": "mb_share_urban",
        "share_agricultura_total": "mb_share_agriculture_total",
    }
    for c_pt, c_en in classes_mb.items():
        if c_pt in mapb.columns:
            mb_keep.append(c_pt)
            mb_rename[c_pt] = c_en
    panel_export = panel_export.merge(
        mapb[mb_keep].rename(columns=mb_rename),
        on=["ibge_code", "year"], how="left",
    )

print(f"Painel final: {panel_export.shape}")

KeyError: 'cultura'

In [ ]:
# Salvar
out_csv = EXPORT_DIR / f"renovabio_outcomes_panel_{DATA_VERSION}.csv"
out_pq = EXPORT_DIR / f"renovabio_outcomes_panel_{DATA_VERSION}.parquet"
panel_export.to_csv(out_csv, index=False, encoding="utf-8")
panel_export.to_parquet(out_pq, index=False)
print(f"OK salvos:")
print(f"  {out_csv.name}  ({out_csv.stat().st_size / 1024:.0f} KB)")
print(f"  {out_pq.name}  ({out_pq.stat().st_size / 1024:.0f} KB)")

## Bloco 3 — Codebooks

In [ ]:
# Codebook PSM
codebook_psm = """# Codebook — RenovaBio PSM Cross-section ({version})

**File:** `renovabio_psm_cross_section_{version}.{{csv,parquet}}`
**Unit of observation:** Municipality
**Universe:** 2,018 municipalities in Brazilian Centro-Sul (states: SP, GO, MG, MS, PR, MT) after B1 filters
**Release:** {date}

## Columns

### Identifiers and treatment status
| Column | Type | Description |
|---|---|---|
| `ibge_code` | string(7) | IBGE municipal code (7-digit), zero-padded |
| `municipality` | string | Municipality name |
| `uf` | string(2) | State (SP, GO, MG, MS, PR, MT) |
| `bioma` | string | Biome (Cerrado, Mata Atlantica, etc.) |
| `is_canavieiro` | boolean | Flag for sugarcane-producing municipality (n=842 universe of paper) |
| `is_treated_ever` | boolean | Ever certified by ANP under RenovaBio (n=194 among canavieiros) |
| `treatment_year` | float | Year of first ANP certification (NaN for never-treated) |

### Demographics and economy (baseline 2015-2019)
| Column | Type | Description | Source |
|---|---|---|---|
| `log_pib_total` | float | log(1 + total GDP, R$ thousand) | IBGE PIB Municipal |
| `log_pib_per_capita` | float | log(1 + GDP per capita, R$) | IBGE PIB Municipal |
| `log_population_2017` | float | log(1 + population 2017) | IBGE Estimativa Populacional |
| `log_total_area_ha` | float | log(1 + total area, hectares) | IBGE |
| `population_density` | float | inhab/ha | derived |
| `share_vabc_agriculture` | float | Agriculture share of municipal value added | IBGE PIB Setorial |
| `share_vabc_industry` | float | Industry share | IBGE PIB Setorial |
| `share_vabc_services` | float | Services share | IBGE PIB Setorial |
| `share_vabc_public_admin` | float | Public administration share | IBGE PIB Setorial |

### MapBiomas baseline (mean 2015-2017)
| Column | Type | Description | Source |
|---|---|---|---|
| `mb_share_sugarcane_baseline` | float | Sugarcane share of municipal area | MapBiomas Collection 9 |
| `mb_share_soybean_baseline` | float | Soybean share | MapBiomas Collection 9 |
| `mb_share_pasture_baseline` | float | Pasture share | MapBiomas Collection 9 |
| `mb_share_native_veg_baseline` | float | Native vegetation share | MapBiomas Collection 9 |
| `mb_share_urban_baseline` | float | Urban/infrastructure share | MapBiomas Collection 9 |
| `mb_share_silviculture_baseline` | float | Silviculture share | MapBiomas Collection 9 |
| `mb_share_agriculture_total_baseline` | float | Total agriculture share | MapBiomas Collection 9 |

### PAM baseline (mean 2015-2017)
| Column | Type | Description | Source |
|---|---|---|---|
| `log_sugarcane_area_ha` | float | log(1 + sugarcane area, ha) | IBGE PAM Tabela 1612 |
| `log_soybean_area_ha` | float | log(1 + soybean area, ha) | IBGE PAM 1612 |
| `log_maize_area_ha` | float | log(1 + maize area, ha) | IBGE PAM 1612 |
| `log_cotton_area_ha` | float | log(1 + cotton area, ha) | IBGE PAM 1612 |
| `log_agriculture_total_area_ha` | float | log(1 + total cropped area) | IBGE PAM 1612 |
| `share_sugarcane_of_agriculture` | float | Sugarcane / total cropped area | derived |

### Censo Agropecuário 2017 — Land structure
| Column | Type | Description | Source |
|---|---|---|---|
| `share_family_farms` | float | Family farms (count) / total farms | IBGE Censo Agro 2017 |
| `share_medium_large_farms` | float | Medium-large farms / total farms | IBGE Censo Agro 2017 |
| `share_family_farms_area` | float | Family farms (area) / total area | IBGE Censo Agro 2017 |
| `share_medium_large_farms_area` | float | Medium-large farms (area) / total area | IBGE Censo Agro 2017 |
| `tractors_per_farm` | float | Mechanization proxy | IBGE Censo Agro 2017 |
| `share_irrigated_farms` | float | Share of farms with irrigation | IBGE Censo Agro 2017 |
| `share_irrigated_area` | float | Share of cropped area irrigated | IBGE Censo Agro 2017 |
| `share_financed_farms` | float | Share of farms with rural credit | IBGE Censo Agro 2017 |
| `share_tech_assistance` | float | Share of farms receiving extension | IBGE Censo Agro 2017 |
| `share_natural_vegetation_area` | float | Natural vegetation share of total area | IBGE Censo Agro 2017 |
| `pct_farms_with_energy` | float | Percent of farms with electricity | IBGE Censo Agro 2017 |

### Social indicators (Atlas Brasil 2017/IDHM, Atlas IVS)
| Column | Type | Description | Source |
|---|---|---|---|
| `hdim_education` | float | HDI education component | Atlas Brasil 2017 |
| `hdim_income` | float | HDI income component | Atlas Brasil 2017 |
| `hdim_longevity` | float | HDI longevity component | Atlas Brasil 2017 |
| `social_vulnerability_infrastructure` | float | Urban infrastructure vulnerability (IVS) | Atlas IVS IPEA |
| `social_vulnerability_human_capital` | float | Human capital vulnerability (IVS) | Atlas IVS IPEA |
| `social_vulnerability_income_work` | float | Income-work vulnerability (IVS) | Atlas IVS IPEA |
| `gini_index` | float | Gini index | Atlas Brasil 2017 |

### Estimated propensity scores (optional, if available)
| Column | Type | Description |
|---|---|---|
| `propensity_score_full2` | float | Estimated propensity score, FULL2 specification (33 covariates) |

## Notes

- All log transformations use `log(1+x)` to handle zeros.
- Share variables are clipped to [0, 1] when 80% of values fall in [-0.05, 1.05].
- Missing values imputed with state-level median in main analysis (not in this exported file).
- For full PSM specification and analysis pipeline, see paper Section 3 and supplementary code.
""".format(version=DATA_VERSION, date=RELEASE_DATE)

(EXPORT_DIR / f"CODEBOOK_PSM_{DATA_VERSION}.md").write_text(codebook_psm)
print(f"OK CODEBOOK_PSM_{DATA_VERSION}.md")

In [ ]:
# Codebook painel
codebook_panel = """# Codebook — RenovaBio Outcomes Panel ({version})

**File:** `renovabio_outcomes_panel_{version}.{{csv,parquet}}`
**Unit of observation:** Municipality × year
**Universe:** 842 sugarcane municipalities (canavieiros) × 13 years (2012-2024) = up to 10,946 rows
**Release:** {date}

## Columns

### Identifiers
| Column | Type | Description |
|---|---|---|
| `ibge_code` | string(7) | IBGE municipal code (7-digit), zero-padded |
| `year` | int | Year of observation |
| `municipality` | string | Municipality name |
| `uf` | string(2) | State |
| `bioma` | string | Biome |
| `is_treated_ever` | boolean | Ever certified by ANP under RenovaBio |
| `treatment_year` | float | Year of first ANP certification |
| `is_post_treatment` | boolean | Is this observation in the post-treatment period for the municipality |

### SEEG emissions — totals (tCO2eq/year, GWP-AR5)
| Column | Type | Description | Source |
|---|---|---|---|
| `emissions_burning_tco2e` | float | Burning of agricultural residues | SEEG Coleção 9 |
| `emissions_soils_managed_tco2e` | float | Managed soils (sum of N inputs, residues) | SEEG Coleção 9 |
| `emissions_luc_tco2e` | float | Land use change | SEEG Coleção 9 |
| `emissions_carbon_soil_tco2e` | float | Soil carbon flux | SEEG Coleção 9 |

### SEEG emissions — sugarcane decomposition (tCO2eq/year, GWP-AR5)
| Column | Type | Description | SEEG Equation |
|---|---|---|---|
| `emissions_sugarcane_residues_tco2e` | float | Residues left after sugarcane harvest (N2O from decomposition) | Eq. 62-66 |
| `emissions_sugarcane_organic_tco2e` | float | Organic inputs from sugarcane (filter cake, vinasse) | Eq. 40, 42 |
| `emissions_sugarcane_direct_tco2e` | float | Sum of residues + organic (consolidated, used as main outcome) | derived |
| `emissions_n_fertilizer_tco2e` | float | Synthetic N fertilizers | Eq. 52-54 |
| `emissions_liming_tco2e` | float | Liming (calcium oxide application) | Eq. 89, 9 |
| `emissions_other_residues_tco2e` | float | Residues other than sugarcane | Eq. 60-61 |

### PAM agricultural production (areas in hectares, production in tonnes)
| Column | Type | Description | Source |
|---|---|---|---|
| `area_sugarcane_ha` | float | Sugarcane cropped area | IBGE PAM 1612 |
| `area_soybean_ha` | float | Soybean cropped area | IBGE PAM 1612 |
| `area_maize_ha` | float | Maize cropped area | IBGE PAM 1612 |
| `area_cotton_ha` | float | Cotton cropped area | IBGE PAM 1612 |
| `production_sugarcane_t` | float | Sugarcane production | IBGE PAM 1612 |
| `production_soybean_t` | float | Soybean production | IBGE PAM 1612 |
| `production_maize_t` | float | Maize production | IBGE PAM 1612 |
| `production_cotton_t` | float | Cotton production | IBGE PAM 1612 |

### MapBiomas land use shares (annual, fraction of total municipal area)
| Column | Type | Description | Source |
|---|---|---|---|
| `mb_share_sugarcane` | float | Sugarcane share | MapBiomas Collection 9 |
| `mb_share_pasture` | float | Pasture share | MapBiomas Collection 9 |
| `mb_share_native_veg` | float | Native vegetation share | MapBiomas Collection 9 |
| `mb_share_soybean` | float | Soybean share | MapBiomas Collection 9 |
| `mb_share_silviculture` | float | Silviculture share | MapBiomas Collection 9 |
| `mb_share_urban` | float | Urban/infrastructure share | MapBiomas Collection 9 |
| `mb_share_agriculture_total` | float | Total agriculture share | MapBiomas Collection 9 |

## Notes

- Emissions are reported in tCO2eq/year using GWP-AR5 conversion factors (SEEG default): CH4=28, N2O=265.
- For GWP-AR6 conversion, multiply N2O-dominated channels by 273/265 = 1.030.
- SEEG sub-channels were extracted via decomposition of original SEEG categories — see paper Section 3 and Brasil (2020e) for methodology.
- MapBiomas shares cover years 2015-2024 (Collection 9 limitation).
- PAM data covers 2012-2024.
- Missing values are not imputed in the exported file.

## Reference equations (SEEG / IPCC Tier 2)

- **Eq. 52**: N fertilizer-N2O = N_applied × EF1 × 44/28 × GWP_N2O
- **Eq. 62-66**: Residue-N2O from sugarcane harvest (depends on %manual harvest with burning)
- **Eq. 40, 42**: Organic N from filter cake and vinasse
- **Eq. 60-61**: Residue-N2O from other crops
- **Eq. 89, 9**: Liming CO2 emissions

Full equations available in Brasil (2020e) - 4° Inventário Nacional de Emissões e Remoções
Antrópicas de Gases de Efeito Estufa: Relatório de Referência — Setor Agropecuária.
""".format(version=DATA_VERSION, date=RELEASE_DATE)

(EXPORT_DIR / f"CODEBOOK_PANEL_{DATA_VERSION}.md").write_text(codebook_panel)
print(f"OK CODEBOOK_PANEL_{DATA_VERSION}.md")

## Bloco 4 — README geral

In [ ]:
readme = """# RenovaBio Causal Impact Dataset — {version}

This dataset accompanies the paper:

> Coutinho, A. L. A. (2026). *Causal impact of RenovaBio biofuel certification on
> municipal agricultural GHG emissions in Brazil's Centro-Sul: a propensity score
> matching + Callaway-Sant'Anna difference-in-differences study*. Submitted to
> Ecological Economics.

**Author:** Alvaro L. A. Coutinho (PPGI-EA, ESALQ/USP — alvaro.coutinho@usp.br)
**Release date:** {date}
**Version:** {version}
**Licence:** CC-BY 4.0 (free use with attribution)

## Files

| File | Description | Rows | Columns |
|---|---|---:|---:|
| `renovabio_psm_cross_section_{version}.csv/parquet` | Cross-section for PSM matching | ~2,018 | ~45 |
| `renovabio_outcomes_panel_{version}.csv/parquet` | Long panel of outcomes | ~10,946 | ~30 |
| `CODEBOOK_PSM_{version}.md` | Variable documentation for PSM | — | — |
| `CODEBOOK_PANEL_{version}.md` | Variable documentation for panel | — | — |

## Quick-start

```python
import pandas as pd

# Load PSM cross-section
psm = pd.read_parquet("renovabio_psm_cross_section_{version}.parquet")
print(psm.shape)  # (~2018, ~45)
print(psm["is_treated_ever"].sum())  # 194 treated municipalities

# Load outcomes panel
panel = pd.read_parquet("renovabio_outcomes_panel_{version}.parquet")
print(panel.shape)  # (~10946, ~30)
panel.groupby("year")["emissions_sugarcane_direct_tco2e"].sum().plot()
```

## Universe

- **Geographic scope:** Brazil Centro-Sul region (states SP, GO, MG, MS, PR, MT)
- **Total municipalities (pre-filter):** ~2,018 in PSM cross-section
- **Sugarcane municipalities (canavieiros):** 842 (analytical universe of the paper)
- **Treated municipalities:** 194 (ever certified by ANP under RenovaBio)
- **Time coverage:**
  - SEEG emissions: 2012–2024 (13 years)
  - PAM agricultural: 2012–2024
  - MapBiomas: 2015–2024 (Collection 9 limit)

## Data sources

This dataset integrates multiple public Brazilian government datasets:

| Source | Data used | Provider |
|---|---|---|
| SEEG Coleção 9 | Municipal GHG emissions by sector and gas | Observatório do Clima / Imaflora |
| IBGE PAM 1612 | Crop areas and production | IBGE |
| IBGE Censo Agro 2017 | Land structure and farm characteristics | IBGE |
| MapBiomas Collection 9 | Annual land use shares | MapBiomas Project |
| ANP Resoluções 22-32/2018 + 758/2018 + posterior | RenovaBio certification dates | ANP |
| IBGE PIB Municipal | Economic indicators | IBGE |
| Atlas Brasil / Atlas IVS | Social indicators (HDI, GINI, IVS) | UNDP / IPEA |

## Methodology

The data integration and analytical pipeline are documented in:
- Paper Section 3 (Methods)
- Pre-registration `preregistro_renovabio_consolidado_v26.md` (request from author)
- Source code (request from author)

## Reproducibility

The dataset is provided in two formats:
- **CSV** for human inspection and language-agnostic analysis
- **Parquet** for efficient analytical workloads (preserves dtypes)

Both versions contain identical data.

## Citation

If you use this dataset, please cite:

```
@article{{coutinho2026renovabio,
  title={{Causal impact of RenovaBio biofuel certification on municipal
         agricultural GHG emissions in Brazil's Centro-Sul}},
  author={{Coutinho, Alvaro L. A.}},
  journal={{Ecological Economics}},
  year={{2026}},
  note={{Dataset version {version}, released {date}}}
}}
```

## Contact

Questions, errata, or collaboration inquiries:
- **Email:** alvaro.coutinho@usp.br
- **Institution:** ESALQ/USP — Piracicaba, SP, Brazil
- **ORCID:** [TO BE FILLED]
- **GitHub:** [TO BE FILLED]

## Changelog

- **{version}** ({date}): initial public release.
""".format(version=DATA_VERSION, date=RELEASE_DATE)

(EXPORT_DIR / "README.md").write_text(readme)
print(f"OK README.md")

## Resumo final

In [ ]:
import os
print("=" * 78)
print("ARQUIVOS EXPORTADOS PARA PUBLICACAO")
print("=" * 78)
print(f"\nDiretorio: {EXPORT_DIR}\n")

for f in sorted(EXPORT_DIR.iterdir()):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:55s} {size_kb:>10.1f} KB")

print()
print("Proximos passos:")
print("  1. Inspecionar visualmente cada arquivo")
print("  2. Validar codebooks (revisar descricoes)")
print("  3. Adicionar ORCID e GitHub no README")
print("  4. Decidir repositorio (Zenodo recomendado para DOI gratis)")
print("  5. Aguardar publicacao do paper para release publico")